In [1]:
"""
Tiny BERT-style Masked Language Model: full, self-contained implementation.

Includes: tokenizer/vocab, dynamic masking (80/10/10), dataset + collate with
padding masks, the model, training loop (AdamW + warmup/decay + grad clipping),
evaluation, and a fill-mask inference helper.

Run:  python tiny_mlm.py
"""

import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# --------------------------------------------------------------------------- #
# Config
# --------------------------------------------------------------------------- #
SEED = 42
D_MODEL = 128
N_HEADS = 4
N_LAYERS = 2
MAX_LEN = 32
BATCH_SIZE = 64
EPOCHS = 15
LR = 1e-3
WARMUP_FRAC = 0.1
MLM_PROB = 0.2  # higher than BERT's 0.15 because sentences are very short
DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

PAD, UNK, MASK, CLS, SEP = "[PAD]", "[UNK]", "[MASK]", "[CLS]", "[SEP]"
SPECIAL_TOKENS = [PAD, UNK, MASK, CLS, SEP]


# --------------------------------------------------------------------------- #
# Toy corpus (synthetic, so the model has real structure to learn)
# --------------------------------------------------------------------------- #
def build_corpus(n_sentences=4000):
    """Animals eat specific foods and live in specific places."""
    facts = {
        "cat": ("fish", "house"),
        "dog": ("bone", "yard"),
        "cow": ("grass", "farm"),
        "rabbit": ("carrot", "burrow"),
        "monkey": ("banana", "jungle"),
        "bird": ("seeds", "nest"),
    }
    adjectives = ["small", "happy", "hungry", "quiet", "big"]
    templates = [
        "the {adj} {animal} eats {food}",
        "the {animal} likes {food} and lives in the {place}",
        "every day the {adj} {animal} finds {food}",
        "the {animal} sleeps in the {place}",
        "a {adj} {animal} lives in the {place} and eats {food}",
    ]
    corpus = []
    for _ in range(n_sentences):
        animal, (food, place) = random.choice(list(facts.items()))
        sent = random.choice(templates).format(
            adj=random.choice(adjectives), animal=animal, food=food, place=place
        )
        corpus.append(sent)
    return corpus


class Vocab:
    def __init__(self, sentences):
        words = sorted({w for s in sentences for w in s.split()})
        self.itos = SPECIAL_TOKENS + words
        self.stoi = {w: i for i, w in enumerate(self.itos)}
        self.pad_id = self.stoi[PAD]
        self.unk_id = self.stoi[UNK]
        self.mask_id = self.stoi[MASK]
        self.cls_id = self.stoi[CLS]
        self.sep_id = self.stoi[SEP]
        self.num_special = len(SPECIAL_TOKENS)

    def __len__(self):
        return len(self.itos)

    def encode(self, sentence, add_special=True):
        ids = [self.stoi.get(w, self.unk_id) for w in sentence.split()]
        if add_special:
            ids = [self.cls_id] + ids + [self.sep_id]
        return ids[:MAX_LEN]

    def decode(self, ids):
        return " ".join(self.itos[i] for i in ids)


# --------------------------------------------------------------------------- #
# Masking
# --------------------------------------------------------------------------- #
def mask_tokens(input_ids, vocab, mlm_prob=MLM_PROB):
    """
    BERT-style dynamic masking. Of the selected positions:
      80% -> [MASK], 10% -> random token, 10% -> left unchanged.
    Returns (masked_input_ids, labels) where labels = -100 at unselected positions.
    """
    input_ids = input_ids.clone()
    labels = input_ids.clone()

    # Never select special tokens or padding
    special = (
        (input_ids == vocab.pad_id)
        | (input_ids == vocab.cls_id)
        | (input_ids == vocab.sep_id)
    )
    prob = torch.full(labels.shape, mlm_prob)
    prob.masked_fill_(special, 0.0)
    selected = torch.bernoulli(prob).bool()
    labels[~selected] = -100  # loss only on selected positions

    # 80%: replace with [MASK]
    to_mask = torch.bernoulli(torch.full(labels.shape, 0.8)).bool() & selected
    input_ids[to_mask] = vocab.mask_id

    # 10%: replace with a random (non-special) token; remaining 10% unchanged
    to_random = (
        torch.bernoulli(torch.full(labels.shape, 0.5)).bool() & selected & ~to_mask
    )
    random_ids = torch.randint(vocab.num_special, len(vocab), labels.shape)
    input_ids[to_random] = random_ids[to_random]

    return input_ids, labels


# --------------------------------------------------------------------------- #
# Dataset / DataLoader
# --------------------------------------------------------------------------- #
class TextDataset(Dataset):
    def __init__(self, sentences, vocab):
        self.examples = [vocab.encode(s) for s in sentences]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


def make_collate_fn(vocab, apply_masking=True):
    def collate(batch):
        max_len = max(len(x) for x in batch)
        input_ids = torch.full((len(batch), max_len), vocab.pad_id, dtype=torch.long)
        for i, ids in enumerate(batch):
            input_ids[i, : len(ids)] = torch.tensor(ids)
        attention_mask = (input_ids != vocab.pad_id).long()  # 1 = real token

        if apply_masking:
            masked_ids, labels = mask_tokens(input_ids, vocab)
        else:
            masked_ids, labels = input_ids, input_ids.clone()
        return masked_ids, attention_mask, labels

    return collate


# --------------------------------------------------------------------------- #
# Model
# --------------------------------------------------------------------------- #
class MLMHead(nn.Module):
    """Prediction head for masked language modeling."""

    def __init__(self, d_model, vocab_size):
        super().__init__()
        self.dense = nn.Linear(d_model, d_model)
        self.activation = nn.GELU()
        self.layer_norm = nn.LayerNorm(d_model)
        self.decoder = nn.Linear(d_model, vocab_size)

    def forward(self, hidden_states):
        x = self.dense(hidden_states)
        x = self.activation(x)
        x = self.layer_norm(x)
        return self.decoder(x)


class TinyMLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=128,
        n_heads=4,
        n_layers=2,
        max_len=128,
        tie_weights=True,
    ):
        super().__init__()
        self.d_model = d_model

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.layer_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(0.1)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=0.1,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=n_layers, enable_nested_tensor=False
        )

        self.mlm_head = MLMHead(d_model, vocab_size)
        if tie_weights:  # share input embedding and output projection
            self.mlm_head.decoder.weight = self.token_emb.weight

        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, input_ids, attention_mask=None):
        """
        input_ids:      (B, L) long
        attention_mask: (B, L) with 1 = real token, 0 = padding (optional)
        returns logits: (B, L, vocab_size)
        """
        _, seq_len = input_ids.shape
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)

        x = self.token_emb(input_ids) + self.pos_emb(positions)
        x = self.layer_norm(x)
        x = self.dropout(x)

        pad_mask = None if attention_mask is None else (attention_mask == 0)
        x = self.encoder(x, src_key_padding_mask=pad_mask)

        return self.mlm_head(x)


# --------------------------------------------------------------------------- #
# Training / evaluation
# --------------------------------------------------------------------------- #
def mlm_loss(logits, labels):
    return F.cross_entropy(
        logits.view(-1, logits.size(-1)), labels.view(-1), ignore_index=-100
    )


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, total_correct, total_masked = 0.0, 0, 0
    for input_ids, attn, labels in loader:
        input_ids, attn, labels = (t.to(DEVICE) for t in (input_ids, attn, labels))
        logits = model(input_ids, attn)
        sel = labels != -100
        n = sel.sum().item()
        if n == 0:
            continue
        total_loss += mlm_loss(logits, labels).item() * n
        total_correct += (logits.argmax(-1)[sel] == labels[sel]).sum().item()
        total_masked += n
    return total_loss / total_masked, total_correct / total_masked


def train(model, train_loader, val_loader, epochs=EPOCHS, lr=LR):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = epochs * len(train_loader)
    warmup_steps = int(WARMUP_FRAC * total_steps)

    def lr_lambda(step):  # linear warmup, then linear decay to 0
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    for epoch in range(1, epochs + 1):
        model.train()
        running, steps = 0.0, 0
        for input_ids, attn, labels in train_loader:
            input_ids, attn, labels = (t.to(DEVICE) for t in (input_ids, attn, labels))
            if (labels != -100).sum() == 0:  # nothing masked -> loss would be NaN
                continue

            logits = model(input_ids, attn)
            loss = mlm_loss(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            running += loss.item()
            steps += 1

        val_loss, val_acc = evaluate(model, val_loader)
        print(
            f"epoch {epoch:2d}/{epochs} | train loss {running / max(1, steps):.4f} "
            f"| val loss {val_loss:.4f} | val masked-acc {val_acc:.3f}"
        )


# --------------------------------------------------------------------------- #
# Inference
# --------------------------------------------------------------------------- #
@torch.no_grad()
def fill_mask(model, vocab, sentence, top_k=3):
    """Predict the word(s) at each [MASK] in `sentence`."""
    model.eval()
    ids = torch.tensor([vocab.encode(sentence)], device=DEVICE)
    attn = torch.ones_like(ids)
    logits = model(ids, attn)

    results = []
    for pos in (ids[0] == vocab.mask_id).nonzero(as_tuple=True)[0].tolist():
        probs = logits[0, pos].softmax(-1)
        probs[: vocab.num_special] = 0.0  # don't predict special tokens
        top = probs.topk(top_k)
        results.append(
            [(vocab.itos[i], round(p, 3)) for p, i in zip(top.values.tolist(), top.indices.tolist())]
        )
    return results


# --------------------------------------------------------------------------- #
# Main
# --------------------------------------------------------------------------- #
def main():
    random.seed(SEED)
    torch.manual_seed(SEED)

    corpus = build_corpus(5000)
    vocab = Vocab(corpus)
    split = int(0.9 * len(corpus))
    train_ds = TextDataset(corpus[:split], vocab)
    val_ds = TextDataset(corpus[split:], vocab)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=make_collate_fn(vocab)
    )
    # Note: validation masking is random each pass, so val metrics vary slightly between epochs
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=make_collate_fn(vocab)
    )

    model = TinyMLM(
        vocab_size=len(vocab),
        d_model=D_MODEL,
        n_heads=N_HEADS,
        n_layers=N_LAYERS,
        max_len=MAX_LEN,
    ).to(DEVICE)

    n_params = sum(p.numel() for p in model.parameters())
    print(f"device: {DEVICE} | vocab size: {len(vocab)} | parameters: {n_params:,}\n")

    train(model, train_loader, val_loader)

    print("\n--- fill-mask demo ---")
    tests = [
        "the cat eats [MASK]",
        "the dog likes bone and lives in the [MASK]",
        "the [MASK] eats banana",
        "a happy rabbit lives in the [MASK] and eats [MASK]",
    ]
    for s in tests:
        print(f"\n{s}")
        for i, preds in enumerate(fill_mask(model, vocab, s)):
            print(f"  mask #{i + 1}: {preds}")

    torch.save(model.state_dict(), "tiny_mlm.pt")
    print("\nsaved weights to tiny_mlm.pt")


if __name__ == "__main__":
    main()

device: cpu | vocab size: 39 | parameters: 422,695

epoch  1/15 | train loss 2.8921 | val loss 1.7761 | val masked-acc 0.554
epoch  2/15 | train loss 1.0608 | val loss 0.6246 | val masked-acc 0.778
epoch  3/15 | train loss 0.5028 | val loss 0.3395 | val masked-acc 0.883
epoch  4/15 | train loss 0.3071 | val loss 0.2343 | val masked-acc 0.906
epoch  5/15 | train loss 0.2573 | val loss 0.2241 | val masked-acc 0.907
epoch  6/15 | train loss 0.2579 | val loss 0.1738 | val masked-acc 0.928
epoch  7/15 | train loss 0.2143 | val loss 0.2014 | val masked-acc 0.919
epoch  8/15 | train loss 0.2126 | val loss 0.2025 | val masked-acc 0.915
epoch  9/15 | train loss 0.2154 | val loss 0.1838 | val masked-acc 0.918
epoch 10/15 | train loss 0.2078 | val loss 0.2558 | val masked-acc 0.903
epoch 11/15 | train loss 0.2008 | val loss 0.1726 | val masked-acc 0.928
epoch 12/15 | train loss 0.2066 | val loss 0.2173 | val masked-acc 0.903
epoch 13/15 | train loss 0.2018 | val loss 0.1928 | val masked-acc 0.915